# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

> This notebook follows FAIR principles and references dataset entities by their `@id`.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}")
print(f"Dataset Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We reference all entities by their `@id`.


In [ ]:
# List all record sets by @id
record_sets = metadata.recordSet if hasattr(metadata, 'recordSet') else []
if len(record_sets) == 0:
    # Try loading recordSet from metadata.to_json()
    raw = dataset.metadata.to_json()
    record_sets = raw.get('recordSet', [])

print("Available record sets (@id):")
record_set_ids = []
for rs in record_sets:
    if isinstance(rs, dict) and '@id' in rs:
        print(f"- {rs['@id']}")
        record_set_ids.append(rs['@id'])
    elif isinstance(rs, str):
        print(f"- {rs}")
        record_set_ids.append(rs)

if len(record_set_ids) == 0:
    print("No record sets found. Check dataset schema.")

# For each record set, print available fields (@id)
for rsid in record_set_ids:
    print(f"\nSample records for record set @id: {rsid}")
    try:
        fields = dataset.fields(record_set=rsid)
        print(f"Fields (@id): {[f['@id'] for f in fields] if fields else 'N/A'}")
        # Show first 2 records
        for i, rec in enumerate(dataset.records(record_set=rsid)):
            print(f"Record {i+1}: {rec}")
            if i >= 1:
                break
    except Exception as e:
        print(f"Failed to load record set {rsid}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

All columns are referenced by their unique `@id`.


In [ ]:
# Example: Load all record sets into DataFrames
# If you want specific record sets, you can filter by their @id as observed above.
dataframes = {}

# Use discovered record_set_ids from previous cell
# If record_set_ids is empty, manually specify if known
if len(record_set_ids) == 0:
    # Example record set @id (Update with actual IDs as available)
    record_set_ids = ['cr:MentalHealthSurveyRecordSet']

for rsid in record_set_ids:
    try:
        records = list(dataset.records(record_set=rsid))
        df = pd.DataFrame(records)
        dataframes[rsid] = df
        print(f"\nDataFrame columns for record set {rsid}:")
        print(df.columns.tolist())
        print("Preview:")
        print(df.head())
    except Exception as e:
        print(f"Could not load record set {rsid}: {e}")

# Select first available DataFrame for further analysis
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]
    df = dataframes[selected_record_set_id]
    print(f"\nUsing record set @id: {selected_record_set_id} for EDA.")
else:
    print("No DataFrames loaded.")

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Reference fields by their `@id`, not by name.


In [ ]:
# EDA on one numeric field (referenced by @id)
import numpy as np

if len(dataframes) > 0:
    df = dataframes[selected_record_set_id]
    print(f"Columns for record set {selected_record_set_id}: {df.columns.tolist()}")

    # Choose GAD-7 Total Score column, which may have @id like 'cr:GAD7TotalScore'
    # If unknown, print columns and pick one numeric
    numeric_field_id = None
    for col in df.columns:
        if 'gad7' in col.lower() or 'phq9' in col.lower() or 'score' in col.lower() or 'cr:GAD7TotalScore' in col:
            # Assume GAD-7 total score @id
            numeric_field_id = col
            break
    if numeric_field_id is None and len(df.columns) > 0:
        # fallback to any numeric column
        for col in df.columns:
            if np.issubdtype(df[col].dtype, np.number) or ('score' in col):
                numeric_field_id = col
                break
    print(f"Using numeric field for EDA: {numeric_field_id}")

    # Set threshold for filtering (e.g., GAD-7 score > 10 indicates moderate/severe anxiety)
    threshold = 10
    if numeric_field_id is not None:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field (e.g., gender or village), referenced by @id
        group_field_id = None
        for col in df.columns:
            if 'gender' in col.lower() or 'cr:gender' in col.lower():
                group_field_id = col
                break
            elif 'village' in col.lower() or 'cr:village' in col.lower():
                group_field_id = col
                break
        print(f"Using group field for grouping: {group_field_id}")

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = (
                filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            )
            print(f"Grouped average {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
else:
    print("No DataFrame to analyze.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of GAD-7 score (referenced by @id)
if len(dataframes) > 0 and numeric_field_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()
    
    # If grouped_df exists, plot group means
    if 'grouped_df' in locals():
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Average {numeric_field_id} by {group_field_id}")
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xlabel(group_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No visualization rendered.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County provides diverse demographic and psychological data, including GAD-7 scores referenced by their `@id`.
- We filtered, normalized, and grouped data for exploratory analysis -- always referencing fields by their Croissant `@id`s.
- Visualizations show the distributions and group averages, supporting further research on mental health patterns in Kenya.

For more advanced analysis or ML tasks, always reference record sets and fields using their `@id`, leveraging the standardized Croissant schema for reproducibility.